# 03 — Model Training

**Purpose:** Train and evaluate the group-stage and knockout models.
Chronological cross-validation, probability calibration, and feature importance analysis.
All model logic lives in `src/models.py` — this notebook drives it and records results.

**Inputs**
- `outputs/training_rows.parquet`
- `outputs/team_features_freeze.parquet`

**Outputs**
- `outputs/group_stage_model.joblib`
- `outputs/knockout_model.joblib`
- `outputs/penalty_model.joblib`
- `outputs/feature_importance_group.csv`
- `outputs/feature_importance_knockout.csv`
- `outputs/cv_results.csv`
- `outputs/charts/calibration_curve.png`

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.leakage_guard import check_no_synthetic_data, check_training_rows_chronological
from src.models import (
    GroupStageModel, KnockoutModel, PenaltyModel,
    ChronologicalCV, compute_rps, compute_brier, compute_directional_accuracy,
    fit_all_models,
    WIN, DRAW, LOSS, CALIBRATION_YEAR,
    PRETOURNAMENT_FEATURES,
)

OUTPUTS = PROJECT_ROOT / "outputs"
CHARTS  = OUTPUTS / "charts"
CHARTS.mkdir(parents=True, exist_ok=True)

print("Imports OK")

## Cell 2 — Load training data

In [ ]:
training_rows  = pd.read_parquet(OUTPUTS / "training_rows.parquet")
team_features  = pd.read_parquet(OUTPUTS / "team_features_freeze.parquet")

print(f"Training rows : {training_rows.shape}")
print(f"Team features : {team_features.shape}")

# Re-run guards on loaded data
check_no_synthetic_data(training_rows)
check_training_rows_chronological(training_rows)
print("✓ Leakage guards re-confirmed on loaded data")

if "outcome" in training_rows.columns:
    dist = training_rows["outcome"].value_counts().sort_index()
    print(f"\nOutcome distribution: {dist.to_dict()}")

## Cell 3 — Chronological CV fold structure

In [ ]:
cv = ChronologicalCV()

if "wc_year" in training_rows.columns:
    fold_sizes = []
    for fold_year, train_years in cv.fold_structure:
        train_mask = training_rows["wc_year"].isin(train_years)
        test_mask  = training_rows["wc_year"] == fold_year
        fold_sizes.append({
            "held_out_year": fold_year,
            "train_rows":    int(train_mask.sum()),
            "test_rows":     int(test_mask.sum()),
        })
    print("CV fold sizes (Fold 7 / 2022 is calibration-only — not in this table):")
    pd.DataFrame(fold_sizes)

## Cell 4 — Train group-stage model with chronological CV

In [ ]:
group_model = GroupStageModel()

print("Fitting GroupStageModel (chronological CV + calibration)...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    cv_results = group_model.fit(training_rows)

print(f"\nGroupStageModel fitted")

if cv_results is not None and hasattr(cv_results, '__iter__'):
    cv_df = pd.DataFrame(cv_results) if not isinstance(cv_results, pd.DataFrame) else cv_results
    print("\nCV results by fold:")
    display(cv_df)

    # Design spec: target accuracy > 69.5%; Elo baseline ≈ 66.7%
    if "accuracy" in cv_df.columns:
        mean_acc = cv_df["accuracy"].mean()
        print(f"\nMean CV accuracy: {mean_acc:.3f}  (target > 0.695, Elo baseline ≈ 0.667)")

## Cell 5 — Calibration analysis on 2022 holdout

In [ ]:
if "wc_year" in training_rows.columns:
    calibration_set = training_rows[training_rows["wc_year"] == CALIBRATION_YEAR].copy()
    print(f"Calibration holdout (2022): {len(calibration_set)} rows")

    feature_cols = [c for c in PRETOURNAMENT_FEATURES if c in calibration_set.columns]
    if feature_cols and "outcome" in calibration_set.columns:
        X_cal = calibration_set[feature_cols]
        y_cal = calibration_set["outcome"].values

        try:
            proba_cal = group_model.predict_proba(X_cal)

            fig, ax = plt.subplots(figsize=(7, 5))
            bins = np.linspace(0, 1, 6)
            bin_centres = (bins[:-1] + bins[1:]) / 2

            # WIN class calibration
            p_win = proba_cal[:, 2]  # WIN is class index 2
            actual_win = (y_cal == WIN).astype(float)
            bin_pred, bin_actual = [], []
            for lo, hi in zip(bins[:-1], bins[1:]):
                mask = (p_win >= lo) & (p_win < hi)
                if mask.sum() > 0:
                    bin_pred.append(p_win[mask].mean())
                    bin_actual.append(actual_win[mask].mean())

            ax.plot(bin_pred, bin_actual, "o-", label="Calibrated model (WIN class)")
            ax.plot([0, 1], [0, 1], "--k", alpha=0.4, label="Perfect calibration")
            ax.set_xlabel("Predicted probability")
            ax.set_ylabel("Actual win rate")
            ax.set_title(f"Calibration curve — 2022 holdout ({len(calibration_set)} rows)")
            ax.legend()
            plt.tight_layout()
            out_path = CHARTS / "calibration_curve.png"
            plt.savefig(out_path, dpi=150)
            plt.show()
            print(f"Saved → {out_path}")
        except Exception as e:
            print(f"Calibration plot skipped: {e}")

## Cell 6 — Train knockout model

In [ ]:
knockout_model = KnockoutModel()

print("Fitting KnockoutModel...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ko_results = knockout_model.fit(training_rows)

print(f"\nKnockoutModel fitted")

# Design spec: Stage Group B (QF/SF/Final) prefers LR over XGBoost
# due to the small (~49 match) training set for later-round matches
if hasattr(knockout_model, "stage_b_primary_model"):
    print(f"Stage B primary model: {type(knockout_model.stage_b_primary_model).__name__}")

## Cell 7 — Train penalty model

In [ ]:
penalty_model = PenaltyModel()

DATA_ROOT = PROJECT_ROOT
ARC3 = DATA_ROOT / "archive (3).zip"
ARC6 = DATA_ROOT / "archive (6).zip"

from src.features import load_ds6, load_ds8
ds6 = load_ds6(ARC6)
ds8 = load_ds8(ARC3)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    penalty_model.fit(ds6, ds8)

print("PenaltyModel fitted")

# Spot-check: Germany vs England
try:
    p_ger, p_eng = penalty_model.predict("Germany", "England")
    assert abs(p_ger + p_eng - 1.0) < 1e-6, "Penalty proba must sum to 1"
    print(f"Germany vs England: P(GER)={p_ger:.3f}  P(ENG)={p_eng:.3f}")
    print(f"✓ P(GER) + P(ENG) = {p_ger+p_eng:.6f}")
except Exception as e:
    print(f"Penalty spot-check skipped: {e}")

## Cell 8 — Feature importance

In [ ]:
for model, name in [(group_model, "group"), (knockout_model, "knockout")]:
    try:
        importance = model.get_feature_importance()
        if importance is not None:
            out_path = OUTPUTS / f"feature_importance_{name}.csv"
            importance.to_csv(out_path)
            print(f"Saved → {out_path}")
            print(f"\nTop 10 {name} features:")
            display(importance.head(10))
    except Exception as e:
        print(f"Feature importance for {name} skipped: {e}")

## Cell 9 — Sanity checks on known 2022 matches

In [ ]:
# Design spec: model should assign >50% win prob to the higher-Elo team in ≥8/10 2022 matches
if "wc_year" in training_rows.columns:
    matches_2022 = training_rows[
        (training_rows["wc_year"] == 2022) &
        (training_rows.get("perspective", pd.Series("home", index=training_rows.index)) == "home")
    ].head(10)

    feature_cols = [c for c in PRETOURNAMENT_FEATURES if c in matches_2022.columns]
    if feature_cols and len(matches_2022) >= 5:
        try:
            proba = group_model.predict_proba(matches_2022[feature_cols])
            p_home_wins = proba[:, 2]  # P(WIN) for home perspective

            directional_ok = 0
            for i, (_, row) in enumerate(matches_2022.iterrows()):
                elo_delta = row.get("elo_rating_delta", 0)
                p_win     = p_home_wins[i]
                # Higher Elo team should be favoured
                if (elo_delta > 0 and p_win > 0.5) or (elo_delta < 0 and p_win < 0.5):
                    directional_ok += 1

            print(f"Directional accuracy on 2022 sample: {directional_ok}/{len(matches_2022)}")
            print("  (design spec target: ≥ 8/10)")
        except Exception as e:
            print(f"Sanity check skipped: {e}")

## Cell 10 — Save models

In [ ]:
models_to_save = {
    "group_stage_model.joblib":  group_model,
    "knockout_model.joblib":     knockout_model,
    "penalty_model.joblib":      penalty_model,
}

for filename, model in models_to_save.items():
    path = OUTPUTS / filename
    joblib.dump(model, path)
    print(f"Saved → {path}")

print("\n✓ All models serialised")